# Imagery feature extraction — shared Python workflow

This notebook calls `pedflow.imagery`; it no longer contains a second extraction implementation.
The original notebook is preserved under `07_test/imagery_before_migration_20260920/`.

The default queue processes up to **10 sites concurrently**, keeping one site's images per LLM request. Completed records are checkpointed and can be resumed. Only `commercial_buildings` and `footpath_both_sides` are exposed to the selected pedestrian model; the full original vision prompt is preserved.

Live execution requires `GOOGLE_MAPS_API_KEY` and `GEMINI_API_KEY` in the root `.env`. Running the extraction cell makes provider requests. Configure request limits for your project in `configs/prediction/imagery_v1.json`.


In [ ]:
from dataclasses import replace
from dotenv import load_dotenv

from pedflow.paths import find_project_root
from pedflow.imagery import load_imagery_config, build_imagery_features
from pedflow.imagery.io import read_sites, write_imagery_outputs

PROJECT_ROOT = find_project_root()
load_dotenv(PROJECT_ROOT / ".env", override=False)
config = load_imagery_config(PROJECT_ROOT / "configs/prediction/imagery_v1.json")
config = replace(config, max_workers=10)


## Input

Use a location table with unique `site_id`, `latitude` and `longitude`; omitted CRS defaults to WGS84. Text addresses must be geocoded upstream. Change the input path and worksheet below for your batch.


In [ ]:
INPUT = PROJECT_ROOT / "data/templates/batch_input_template.xlsx"
SHEET = "Batch_Input"
OUTPUT = PROJECT_ROOT / "outputs/imagery_features/notebook_run"
CHECKPOINT = OUTPUT / "imagery.checkpoint.json"

sites = read_sites(INPUT, sheet=SHEET)
display(sites.head())


## Extract or resume

Completed and needs-review records are reused; failed requests are retried. A changed input set, coordinates, model, prompt or sampling configuration requires a new checkpoint path. Changes to concurrency or request pacing do not invalidate completed records.


In [ ]:
result = build_imagery_features(
    sites, config, checkpoint_path=CHECKPOINT, resume=CHECKPOINT.exists(),
    progress=lambda done, total, site, status: print(f"{done}/{total} {site}: {status}"),
)
display(result.features)


In [ ]:
written = write_imagery_outputs(result, OUTPUT)
print(written)
print(result.metadata["status_counts"])


## Interpretation and remaining validation

`complete` means the two requested features passed type checks, not that a human verified them. `needs_review` retains undetermined values as missing. `failed` or `no_imagery` also preserve the site and never substitute zero/False. Review warnings such as aerial-only coverage before using results.

The original prompt, satellite zoom and Street View headings are retained. The nominal 50-metre offset remains a Web Mercator coordinate offset, not a geodesic ground-distance radius. No new historical image-date selector or road-orientation rule has been introduced.

The existing training workbook records incomplete provenance for the supplied historical imagery CSV. The migrated recipe therefore needs a reviewed comparison with training sites before claiming production feature equivalence. LLM outputs can vary across repeated calls, and different image dates/views can change feature values. Passing offline migration tests does not establish imagery accuracy or new-road predictive performance.

See `docs/02_feature_extraction/imagery_integration.md` for configuration, statuses, retries, output files and source references.
